---
title: Research of population features 2024
date: now
author: Jan Cap
---

In [ ]:
from geoscore_de.data_flow.features.municipality import MunicipalityFeature

municipalities = MunicipalityFeature("../../data/raw/municipalities_2022.csv")
municipalities = municipalities.load_transform()
municipalities.columns

In [ ]:
# raw data
from geoscore_de.data_flow.features.population import PopulationFeature

population_feature = PopulationFeature(
    "../../data/raw/features/12411-02-03-5-population.csv", "../../data/tform/features/population.csv"
)
population_feature.load()

Population data contain one row per municipality and age group. One row contains people count, male count, and female count. Columns were translated to english.
To transform the data to required format, the pivoting was used to create separate columns for each age group.

## Municipality alignment

In [ ]:
population = population_feature.load_transform()
# merge population and municipalities on municipality ID
population = population.merge(municipalities, on="AGS", how="outer", indicator=True)
population

In [ ]:
population["_merge"].value_counts()

Population data contain all municipalities but some of them are not present in the municipality data. That is due to the fact that population data contain more than just municipalities, but also districts of larger geographical units. We can safely ignore those rows, because they are just grouping of municipalities and do not contain any new information.

In [ ]:
population = population_feature.load_transform()
# merge population and municipalities on municipality ID
population = population.merge(municipalities, on="AGS", how="right")
population

In [ ]:
population[population["total_population"].isna()]

There is 35 rows with missing population data, although they are present in both datasets. The problem is that population data counts are missing in the source data for those municipalities, so we cannot do anything about it. We will just have to ignore those municipalities in the analysis.

## Data Exploration

In [ ]:
population.dtypes

In [ ]:
print("Population according to 2022 municipality data:", population["Persons"].sum())
print("Population according to 2024 population data:", int(population["total_population"].sum()))

In [ ]:
# difference between 2022 municipality data and 2024 population data
population["people_diff"] = population["total_population"] - population["Persons"]

In [ ]:
print("Between 2022 and 2024, population increased by", population["people_diff"].sum())

In [ ]:
import plotnine as gg
from mizani.formatters import comma_format
from plotnine import aes, geom_histogram, ggplot, labs, theme_minimal

(
    ggplot(
        population.dropna(subset=["people_diff"]),
        aes(x="people_diff"),
    )
    + geom_histogram(bins=40, color="white")
    + labs(
        title="Distribution of Population Difference between 2022 and 2024 data",
        x="Population difference (log scale)",
        y="Number of municipalities",
    )
    + gg.scale_x_log10(labels=comma_format())
    + theme_minimal()
)

Looks like almost all municipality increased in population slightly. Lets look at percentage change between 2022 municipality data and 2024 population data.

In [ ]:
# percentage change between 2022 municipality data and 2024 population data
population["people_diff_pct"] = population["people_diff"] / population["Persons"]
display(
    population[["AGS", "total_population", "Persons", "people_diff_pct"]]
    .sort_values("people_diff_pct", ascending=False)
    .head(10)
)
display(population[["AGS", "total_population", "Persons", "people_diff_pct"]].sort_values("people_diff_pct").head(10))

In [ ]:
(
    ggplot(
        population.dropna(subset=["people_diff_pct"]),
        aes(x="people_diff_pct"),
    )
    + geom_histogram(bins=40, color="white")
    + labs(
        title="Distribution of Percentage Population Difference between 2022 and 2024 data",
        x="Population difference",
        y="Number of municipalities",
    )
    + theme_minimal()
)

Percentage change is small in boundaries of -100% and 250%. 

Next think to explore is distribution of municipality sizes.

In [ ]:
import plotnine as gg
from mizani.formatters import comma_format
from plotnine import aes, geom_histogram, ggplot, labs, theme_minimal

(
    ggplot(
        population.dropna(subset=["total_population"]),
        aes(x="total_population"),
    )
    + geom_histogram(bins=40, color="white")
    + labs(
        title="Distribution of municipality size",
        x="Total population (log scale)",
        y="Number of municipalities",
    )
    + gg.scale_x_log10(labels=comma_format())
    + theme_minimal()
)

Municipalities look normal distributed on log x scale, which meens that there are many small municipalities and few large ones.

Let's explore man/female proportions in municipalities.

In [ ]:
population

In [ ]:
print(
    "Average male proportion",
    (population["male_proportion"] * population["total_population"]).sum() / population["total_population"].sum(),
)
(
    ggplot(
        population.dropna(subset=["male_proportion"]),
        aes(x="male_proportion"),
    )
    + geom_histogram(bins=40, color="white")
    + labs(
        title="Distribution of male proportion",
        x="Male proportion",
        y="Number of municipalities",
    )
    # add x axis limits to 0 and 1
    + gg.xlim(0, 1)
    + theme_minimal()
)

In [ ]:
import matplotlib.pyplot as plt

from geoscore_de.data_flow.geo import load_geo_data

geojson = load_geo_data("../../data/georef-germany-gemeinde.csv")

In [ ]:
geojson_population = geojson.merge(
    population[["AGS", "male_proportion", "total_population"]],
    on="AGS",
    how="left",
)

fig, ax = plt.subplots(1, 1, figsize=(12, 8))

geojson_population.plot(
    column="male_proportion",
    ax=ax,
    cmap="RdBu",
    linewidth=0.05,
    edgecolor="white",
    legend=True,
    missing_kwds={"color": "lightgrey", "label": "No data"},
    vmin=0.4,
    vmax=0.6,
    legend_kwds={"shrink": 0.7},
)

plt.title("Male Proportion in German Municipalities")
plt.show()

In [ ]:
display(
    geojson_population.sort_values("male_proportion")[
        ["Gemeinde name", "AGS", "total_population", "male_proportion"]
    ].head(10)
)
display(
    geojson_population.sort_values("male_proportion", ascending=False)[
        ["Gemeinde name", "AGS", "total_population", "male_proportion"]
    ].head(10)
)

Most of the municipalities is around 50%, but there are some with very low or very high male proportions.
Especially interesting is Osterheide with only 13% of males. Today Osterheide covers the NATO Bergen-Hohne firing ranges in Heidekreis district which were established in 1958.
Based on following age analyses this municipality has significant share of people over 65 years, which can be explained by the fact that many people who worked in the firing ranges and their families lived there. After the end of the Cold War, many of these people moved away, leaving behind a population that is older and has a higher proportion of females.

In [ ]:
age_share_groups = {
    "share_under_18": [
        "age_under_3",
        "age_3_to_5",
        "age_6_to_9",
        "age_10_to_14",
        "age_15_to_17",
    ],
    "share_working_age": [
        "age_18_to_19",
        "age_20_to_24",
        "age_25_to_29",
        "age_30_to_34",
        "age_35_to_39",
        "age_40_to_44",
        "age_45_to_49",
        "age_50_to_54",
        "age_55_to_59",
        "age_60_to_64",
    ],
    "share_65_plus": ["age_65_to_74", "age_75_and_over"],
}

population_map = population.copy()
for share_column, age_columns in age_share_groups.items():
    population_map[share_column] = population_map[age_columns].sum(axis=1, min_count=1)

geojson = load_geo_data("../../data/georef-germany-gemeinde.csv")

geojson_population = geojson.merge(
    population_map[["AGS", "share_under_18", "share_working_age", "share_65_plus"]],
    on="AGS",
    how="left",
)

fig, axes = plt.subplots(1, 3, figsize=(21, 8))
plot_specs = [
    ("share_under_18", "Proportion of Population under 18"),
    ("share_working_age", "Proportion of Working-age Population (18-64)"),
    ("share_65_plus", "Proportion of Population 65+"),
]

for ax, (column, title) in zip(axes, plot_specs):
    geojson_population.plot(
        column=column,
        ax=ax,
        cmap="YlOrRd",
        linewidth=0.05,
        edgecolor="white",
        legend=True,
        missing_kwds={"color": "lightgrey", "label": "No data"},
        # vmin=0,
        # vmax=1,
        legend_kwds={"shrink": 0.7},
    )
    ax.set_title(title)
    ax.set_axis_off()

plt.tight_layout()
plt.show()

From the age proportion analyses we can see difference between east and west germany. In east germany there are more municipalities with higher share of people over 65 years, which can be explained by the fact that many young people moved away from east germany after the reunification, leaving behind an older population. In west germany there are more municipalities with higher share of people under 18 years and working age population.